# Récupération des données Odissé



In [11]:
import pandas as pd

In [ ]:
### Code de récupération des données sur le site
# à résoudre : problème de clé API

import requests
import os
api_key = os.environ.get("API_KEY")
headers = {
    "Authorization": f"Apikey {api_key}",
    "Content-Type": "application/json"
}
response = requests.get("https://odisse.santepubliquefrance.fr/api/explore/v2.1/catalog/datasets/traumatisme-passages-aux-urgences-et-actes-sos-medecins-departement/records/?lang=fr&limit=10&offset=0", headers=headers)

In [ ]:
### En passant par le format parquet depuis un bucket personnel
import pyarrow.parquet as pq

# Définir le fichier de destination
fichier_bobos = "s3://jvalade/traumatisme-passages-aux-urgences-et-actes-sos-medecins-departement.parquet"

# Charger le fichier Parquet
table = pq.read_table(fichier_bobos)
bobos = table.to_pandas()

In [6]:
bobos.head()

,date_complet,semaine,dep,libgeo,sursaud_cl_age_gene,taux_passages_trauma_sau,taux_hospit_trauma_sau,taux_actes_trauma_sos,reg,reglib
0,2026-10-12,2026-S42,89,Yonne,15-64 ans,28571.428571,13815.789474,2197.802198,27,Bourgogne et Franche-Comté
1,2026-10-12,2026-S42,91,Essonne,65 ans ou plus,17247.386760,12621.359223,2793.296089,11,Île-de-France
2,2026-10-12,2026-S42,92,Hauts-de-Seine,00-14 ans,29592.621061,8530.805687,NaN,11,Île-de-France
3,2026-10-12,2026-S42,92,Hauts-de-Seine,15-64 ans,22916.304159,5475.040258,NaN,11,Île-de-France
4,2026-10-12,2026-S42,92,Hauts-de-Seine,Tous âges,23935.288958,8652.575958,NaN,11,Île-de-France


In [7]:
bobos["sursaud_cl_age_gene"].unique()

<ArrowStringArray>
['15-64 ans', '65 ans ou plus', '00-14 ans', 'Tous âges']
Length: 4, dtype: str

In [ ]:
### Détermination de la plage des données

print(bobos["date_complet"].min())
print(bobos["date_complet"].max())

2021-01-04
2026-10-12


Note : il manque le nombre total d'hospitalisations pour qu'on puisse considérer ces données (on a uniquement le taux d'hospitalisation pour traumatisme, on peut en dire des choses mais à la fois...)

Pour parer à ce problème on peut utiliser le jeu de données de l'enquête de la DREES : « Passages aux urgences entre 2017 et 2023 : des dynamiques contrastées selon les départements »

In [48]:
### Import des données de l'enquête de la DREES par API : 
import requests

limit = 100
url_api_drees = f'https://data.drees.solidarites-sante.gouv.fr/api/explore/v2.1/catalog/datasets/series_longues_passages_urgences_2017_2023/records?limit={limit}'

req = requests.get(url_api_drees)
print(req)


<Response [200]>


Note : l'API limite le nombre de résultats à 100, ça risque d'être contraignant...

In [49]:
nb_hospi = pd.DataFrame(req.json()["results"])
print(len(nb_hospi))
nb_hospi.head()


100


,date,dep,libelle_dep,nb_passages
0,2017-12-25,61,Orne,397.1
1,2018-01-29,61,Orne,429.6
2,2018-02-18,61,Orne,378.7
3,2018-03-13,61,Orne,385.8
4,2018-04-10,61,Orne,352.6


Mais nous pouvons télécharger le fichier parquet complet : 

In [51]:
### En passant par le format parquet depuis un bucket personnel
import pyarrow.parquet as pq

# Définir le fichier de destination
fichier_hospi = "s3://jvalade/series_longues_passages_urgences_2017_2023.parquet"

# Charger le fichier Parquet
table = pq.read_table(fichier_hospi)
nb_hospi = table.to_pandas()

In [54]:
print(len(nb_hospi))
nb_hospi.head(30)


247933


,date,dep,libelle_dep,nb_passages
0,2017-12-25,61,Orne,397.1
1,2018-01-29,61,Orne,429.6
2,2018-02-18,61,Orne,378.7
3,2018-03-13,61,Orne,385.8
4,2018-04-10,61,Orne,352.6
5,2018-04-19,61,Orne,383.0
6,2018-05-02,61,Orne,404.8
7,2018-05-15,61,Orne,376.0
8,2018-06-20,61,Orne,408.6
9,2018-07-20,61,Orne,409.6


In [130]:
### Calcul de la durée entre deux relevés
# En sélectionnant pour un département, exemple pour la Réunion
import numpy as np

hospi_run = nb_hospi[nb_hospi["dep"] == '974']

hospi_run["date_bis"] = pd.to_datetime(hospi_run["date"])
hospi_run = hospi_run.sort_values("date_bis")
hospi_run = hospi_run.reset_index(drop = True)
print(hospi_run.head())

hospi_run["delai_entre_deux_obs"] = hospi_run["date_bis"][1:].reset_index(drop = True) - hospi_run["date_bis"]
print(hospi_run["delai_entre_deux_obs"].min())
print(hospi_run["delai_entre_deux_obs"].max())
print(hospi_run["delai_entre_deux_obs"].unique())

         date  dep libelle_dep  nb_passages   date_bis
0  2018-01-01  974  La Réunion        557.0 2018-01-01
1  2018-01-02  974  La Réunion        554.1 2018-01-02
2  2018-01-03  974  La Réunion        481.5 2018-01-03
3  2018-01-04  974  La Réunion        460.2 2018-01-04
4  2018-01-05  974  La Réunion        488.3 2018-01-05
1 days 00:00:00
1 days 00:00:00
<TimedeltaArray>
['1 days', NaT]
Length: 2, dtype: timedelta64[s]
